In [ ]:
from isort.core import process
import pandas as pd
from pathlib import Path
%matplotlib inline
%load_ext autoreload
%autoreload 2
from imports import *
import scipy.io
from config import dir_config, ephys_config
from src.utils import ephys_utils
import pickle
from scipy.stats import ttest_rel
import pyarrow.csv as pa_csv
import pyarrow.parquet as pq
import pyarrow as pa

compiled_dir = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)

In [ ]:
session_metadata = pd.read_csv(processed_dir / "sessions_metadata.csv")

## Parquet conversion

Run once per session. Parquet loads ~10x faster and uses ~3x less disk space.


In [ ]:
def csv_to_parquet(csv_path: Path, chunksize: int = 1_000_000) -> Path:
    parquet_path = csv_path.with_suffix(".parquet")
    if parquet_path.exists():
        print(f"Already exists, skipping: {parquet_path.name}")
        return parquet_path

    print(f"Converting {csv_path.name} ({csv_path.stat().st_size / 1e9:.1f} GB) ...")

    convert_opts = pa_csv.ConvertOptions(
        column_types={"eye_x": pa.float32(), "eye_y": pa.float32()}
    )
    read_opts = pa_csv.ReadOptions(block_size=chunksize * 30)  # ~bytes per chunk

    writer = None
    with pa_csv.open_csv(csv_path, read_options=read_opts, convert_options=convert_opts) as reader:
        for i, batch in enumerate(reader):
            if writer is None:
                writer = pq.ParquetWriter(parquet_path, batch.schema)
            writer.write_batch(batch)
            print(f"  chunk {i+1} written", end="\r")
    if writer:
        writer.close()

    print(f"\nDone -> {parquet_path.name}")
    return parquet_path


def convert_eye_to_degree(eye_data, offset_x=0, offset_y=0, gain_x=1, gain_y=1, rotation=0):    
    # Extract as float32 numpy arrays to avoid pandas/numexpr upcasting to float64
    ex = eye_data.eye_x.to_numpy(dtype=np.float32, copy=True)
    ey = eye_data.eye_y.to_numpy(dtype=np.float32, copy=True)

    # Center and apply gain in-place
    ex -= np.float32(offset_x)
    ey -= np.float32(offset_y)
    ex *= np.float32(gain_x)
    ey *= np.float32(gain_y)

    # Rotate in-place using one temp buffer (avoids two extra full arrays)
    cos_theta = np.float32(np.cos(np.radians(rotation)))
    sin_theta = np.float32(np.sin(np.radians(rotation)))
    ex_copy = ex.copy()
    ex *= cos_theta
    ex -= ey * sin_theta      # rotated_x = ex*cos - ey*sin
    ey *= cos_theta
    ey += ex_copy * sin_theta  # rotated_y = ex*sin + ey*cos

    eye_data.eye_x = ex
    eye_data.eye_y = ey



In [ ]:
# loop over sessions and process eye data
for _, session_row in session_metadata.iterrows():    
    session_id = session_row.session_id
    # convert eye tracking csv to parquet
    eye_filename = compiled_dir / session_id / f"{session_id}_eye_tracking.csv"
    eye_parquet = csv_to_parquet(eye_filename) 

## Process (x and y offset)

In [ ]:
# loop over sessions and process eye data
for _, session_row in session_metadata.iterrows():
    session_id = session_row.session_id
    eye_filename = compiled_dir / session_id / f"{session_id}_eye_tracking.parquet"
    # process the eye data (center, apply gain, rotate) if not already processed
    processed_parquet = eye_filename.with_name(eye_filename.stem + "_processed.parquet")
    if processed_parquet.exists():
        print(f"Already processed, skipping: {processed_parquet.name}")
        continue

    # read eye_data and timestamps
    eye_data = pd.read_parquet(eye_filename, engine="fastparquet")
    timestamp_filename = compiled_dir / session_id / f"{session_id}_timestamps.csv"
    timestamps = pd.read_csv(timestamp_filename)
    
    ts = eye_data.timestamp.values 
    ex = eye_data.eye_x.values
    ey = eye_data.eye_y.values

    # window to use
    window_length = 200 * 30  # 200 ms at 30 kHz
    target_onset = timestamps.target_onset.values # aligned to target onset

    # if timestamps.response_onset is nan(invalid trials) then remove the corresponding target_onset
    valid_target_onset = target_onset[~timestamps.response_onset.isna().values]

    starts = np.searchsorted(ts, valid_target_onset)
    # build index matrix: shape (n, window_length)
    idx = starts[:, None] + np.arange(window_length)[None, :]

    valid = idx < len(ts)
    idx_clipped = np.where(valid, idx, 0)

    wx = ex[idx_clipped]  # (n, 200 * 30)
    wy = ey[idx_clipped]
    wx[~valid] = np.nan
    wy[~valid] = np.nan

    offset_x = np.nanmean(wx)
    offset_y = np.nanmean(wy)

    print(f"Session: {session_id}, Offset X: {offset_x:.2f}, Offset Y: {offset_y:.2f}")
    convert_eye_to_degree(eye_data, offset_x=offset_x, offset_y=offset_y) # offset x and y by average fixation position
    # save the processed eye data as a new parquet file
    eye_data.to_parquet(processed_parquet, engine="fastparquet", index=False)
    print(f"Processed eye data saved to: {processed_parquet.name}")